# Cell-Cell Interaction Analysis with Visium HD Data

This tutorial runs through stLearn CCI analysis on extremely large data, containing >583,000 8μm binned squares (when DEMO = False) with gene expression measured in space.

### Environment setup

In [1]:
import os
import platform
import random

# Make sure all the seeds are set
seed = 0
random.seed(seed)

# Only constrain threads on macOS where BLAS/numba deadlocks are common.
# Must run before any numpy/scanpy import.
if platform.system() == "Darwin":
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["NUMBA_NUM_THREADS"] = "1"
    n_cpus = 1
else:
    n_cpus = None

In [2]:
import stlearn
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import spatialdata

stlearn.settings.set_figure_params(dpi=120)
np.random.seed(seed)

# Ignore all warnings
import warnings

warnings.filterwarnings("ignore")

### Loading the Data

In [ ]:
# Setup download directory and get data
stlearn.settings.datasetdir = pathlib.Path.cwd().parent / "data"
sample_id = "Visium_HD_FF_Human_Breast_Cancer"
bin_size = 8
table_name = f"square_{bin_size:03d}um"
data_dir = stlearn.settings.datasetdir / sample_id
DEMO = True

sdata = stlearn.datasets.visium_hd_sge(sample_id=sample_id, bin_size=bin_size, include_hires_tiff=True)
adata = sdata.tables[table_name]

In [ ]:
# Crop the sample to the center quarter using a bounding box.
if DEMO:
    sdata = stlearn.io.store_checkpoint(data_dir / f"{sample_id}_{table_name}_demo.zarr", lambda: stlearn.io.crop(sdata))
adata = sdata.tables[table_name]
print(f"{adata.n_obs:,} bins x {adata.n_vars:,} genes")

In [ ]:
# QC - Filter genes and cells with at least 10 counts
clustered_name = f"{table_name}_clustered"

def cluster():
    adata = sdata.tables[table_name].copy()
    stlearn.pp.filter_genes(adata, min_counts=10)
    stlearn.pp.filter_cells(adata, min_counts=10)
    adata.layers["counts"] = adata.X.copy()
    stlearn.pp.normalize_total(adata)
    stlearn.pp.log1p(adata)
    stlearn.pp.highly_variable_genes(adata, n_top_genes=2000)
    stlearn.em.run_pca(adata, n_comps=50, random_state=seed)
    stlearn.pp.neighbors(adata, n_neighbors=25, use_rep="X_pca", random_state=seed)
    stlearn.tl.clustering.leiden(adata, resolution=1.05, random_state=seed)
    return adata

adata = stlearn.io.checkpoint(sdata, clustered_name, cluster)

In [ ]:
stlearn.pl.sdata_plot(sdata, "leiden", table_key=clustered_name)

## Note on normalisation

Resets to raw counts - no log1p or shrinking to make genes of similar expression range. In our case, for calling hotspots, we want genes to be more separate, since we select background genes with similar expression levels to detect hotspots.

In [ ]:
adata.X = adata.layers["counts"].copy()
stlearn.pp.normalize_total(adata)

## Gridding

Now performing the gridding. The resolution chosen here may affect the results. The higher resolution, the better this represents the single cell data but the longer the computation takes.

To summarise the gene expression across cells in a grid, we sum the library size normalised gene expression. Summing allows for representing the fact there are multiple cells in a given spot.

In [ ]:
stlearn.io.attach_spatial(adata, data_dir / "binned_outputs" / table_name / "spatial", library_id=sample_id)

In [ ]:
### Calculating the number of grid spots we will generate
# Grid squares of about 52 µm, close to a Visium spot (55 µm): 125 by 125 on the full
# 6.5 mm capture area, and the same spacing on the demo region.
scalefactors = adata.uns["spatial"][sample_id]["scalefactors"]
hires_px_per_um = scalefactors["tissue_hires_scalef"] / scalefactors["microns_per_pixel"]
extent_um = (adata.obsm["spatial"].max(0) - adata.obsm["spatial"].min(0)) * scalefactors["microns_per_pixel"]
n_col, n_row = np.round(extent_um / 52).astype(int)
print(f'{n_row} by {n_col} has this many spots:\n', n_row * n_col)

By providing 'use_label' to the function below, the cell type information is saved as deconvolution information
per spot, and also the dominant cell annotation. That way we can perform stLearn CCI with the cell type information!

In [ ]:
### Gridding.
grid_name = f"grid_{n_row}x{n_col}"
grid = stlearn.io.checkpoint(sdata, grid_name, lambda:
    stlearn.tl.cci.grid(adata, n_row=n_row, n_col=n_col, use_label='leiden', n_cpus=n_cpus))

### Checking the gridding

Comparing the gridded data to the original, to make sure it makes sense.

It's recommended to visualise the dominant cell types per spot, in order to gauge whether the tissue structure is adequately maintained after gridding (i.e. to make sure it is not too low resolution!).

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(20, 5))
stlearn.pl.cluster_plot(grid, use_label='leiden', size=10, ax=axes[0], show_plot=False, show_image=False)
stlearn.pl.cluster_plot(adata, use_label='leiden', ax=axes[1], show_plot=False, show_image=False)
axes[0].set_title(f'Grid leiden dominant spots')
axes[0].set_aspect('equal')
axes[1].set_title(f'Bins leiden labels')
axes[1].set_aspect('equal')
plt.show()

In [ ]:
# Clusters 5/0/1 matched to DCIS/immune/stroma by markers for this data and seed.
groups = list(grid.obs['leiden'].cat.categories)
dcis_cluster = groups[5]
immune_cluster = groups[0]
stroma_cluster = groups[1]
for group in [dcis_cluster, immune_cluster, stroma_cluster]:
    fig, axes = plt.subplots(ncols=3, figsize=(20, 5))
    grid.obs['group'] = grid.uns['leiden'][group].values
    stlearn.pl.feat_plot(grid, feature='group', ax=axes[0], show_plot=False, vmax=1, show_color_bar=False,
                         show_image=False)
    stlearn.pl.cluster_plot(grid, use_label='leiden', list_clusters=[group], ax=axes[1], show_plot=False,
                            show_image=False)
    stlearn.pl.cluster_plot(adata, use_label='leiden', list_clusters=[group], ax=axes[2], show_plot=False,
                            show_image=False)
    axes[0].set_title(f'Grid {group} proportions (max = 1)')
    axes[0].set_aspect('equal')
    axes[1].set_title(f'Grid {group} max spots')
    axes[1].set_aspect('equal')
    axes[2].set_title(f'Individual bin {group}')
    axes[2].set_aspect('equal')
    plt.show()

In [ ]:
gene_name = 'ITGB6'
interesting_lr = 'FN1_ITGB6'

x = adata.obs_vector(gene_name)
vmax = np.percentile(x[x > 0], 99)

fig, axes = plt.subplots(ncols=2, figsize=(20, 8))
stlearn.pl.gene_plot(grid, gene_symbols=gene_name, ax=axes[0], show_color_bar=False, show_plot=False, show_image=False)
stlearn.pl.gene_plot(adata, gene_symbols=gene_name, ax=axes[1], show_color_bar=False, show_plot=False, vmax=vmax, show_image=False)
axes[0].set_aspect('equal')
axes[1].set_aspect('equal')
axes[0].set_title(f'Grid {gene_name} expression')
axes[1].set_title(f'Binned {gene_name} expression')
plt.show()

## LR Permutation Test

Running the LR permutation test, to determine regions of high LR co-expression.

In [ ]:
# Loading the LR databases available within stlearn (from NATMI)
lrs = stlearn.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')
print(len(lrs))

In [ ]:
# Running the analysis #
# stLearn measures `distance` in pixels of the hires image, so set it in microns and convert:
# 270 µm is 250 hires pixels on this sample, about five grid squares.
distance = 270 * hires_px_per_um
lr_name = f"{grid_name}_lr"
lr_results = stlearn.io.checkpoint(sdata, lr_name, compute=lambda:
    stlearn.tl.cci.run(grid, lrs,
                       min_spots=20,       # Filter out any LR pairs with no scores for less than min_spots
                       distance=distance,  # None defaults to spot+immediate neighbours; distance=0 for within-spot mode
                       n_pairs=1000,       # Number of random pairs to generate; low as example, recommend ~10,000
                       n_cpus=n_cpus,      # Number of CPUs for parallel. If None, detects & use all available.
                       random_state=seed
                       ))
stlearn.io.save_table(sdata, grid_name)  # cci.run added the neighbours to grid

In [ ]:
### Can adjust significance thresholds.
stlearn.tl.cci.adj_pvals(grid, correct_axis='spot', pval_adj_cutoff=0.05, adj_method='fdr_bh')

In [ ]:
lr_info = grid.uns['lr_summary']  # A dataframe detailing the LR pairs ranked by number of significant spots.
lr_info

In [ ]:
# Showing the rankings of the LR from a global and local perspective.
# Ranking based on number of significant hotspots.
stlearn.pl.lr_summary(grid, n_top=500)
stlearn.pl.lr_summary(grid, n_top=50, figsize=(10, 3))

### Downstream visualisations from LR analysis

For more downstream visualisations, please see the stLearn CCI tutorial:

https://stlearn.readthedocs.io/en/latest/tutorials/cell_cell_interaction.html

In [ ]:
stats = ['lr_scores', '-log10(p_adjs)', 'lr_sig_scores']
fig, axes = plt.subplots(ncols=len(stats), figsize=(20, 8))
for i, stat in enumerate(stats):
    stlearn.pl.lr_result_plot(grid, use_result=stat, use_lr=interesting_lr, show_color_bar=False, ax=axes[i], show_image=False)
    axes[i].set_aspect('equal')
    axes[i].set_title(f'{interesting_lr} {stat}')

## Predicting Significant Cell-Cell Interactions

For this analysis, we are using distance mode, with the deconvolution information which is stored in grid.uns, as automatically determined by stlearn.tl.cci.grid. The permutation is performed to permute the cell proportions associated with each spot, to determine spots located in LR hotspots that have certain cell types over-represented.

In [ ]:
stlearn.tl.cci.run_cci(grid,
                       'leiden',              # Cell information either in data.obs or data.uns
                       min_spots=2,           # Minimum number of spots for LR to be tested.
                       spot_mixtures=True,    # If True will use the deconvolution data,
                       # so spots can have multiple cell types if score>cell_prop_cutoff
                       cell_prop_cutoff=0.1,  # Spot considered to have cell type if score>0.1
                       sig_spots=True,        # Only consider neighbourhoods of spots which had significant LR scores.
                       n_perms=100,           # Permutations of cells to get background, recommend ~1000
                       random_state=seed,
                       n_cpus=n_cpus,
                       )

## Diagnostic: checking for interaction and cell type frequency correlation

Should be little to no correlation; indicating the permutation has adequately controlled for cell type frequency.

In [ ]:
stlearn.pl.cci_check(grid, 'leiden', figsize=(16, 5))

## CCI Visualisations

### CCI Network Plot

In [ ]:
pos_1 = stlearn.pl.ccinet_plot(grid, 'leiden', return_pos=True, min_counts=30)
stlearn.pl.ccinet_plot(grid, 'leiden', interesting_lr, min_counts=2,
                       figsize=(10, 7.5), pos=pos_1)

### CCI Chord Plot

In [ ]:
stlearn.pl.lr_chord_plot(grid, 'leiden')
stlearn.pl.lr_chord_plot(grid, 'leiden', interesting_lr)